# Appendix A: Introduction to PyTorch

##### Listing A.1 Creating PyTorch tensors

In [2]:
# Create tensors with different ranks to inspect their shapes and contents.
import torch

tensor0d = torch.tensor(1)

tensor1d = torch.tensor([1, 2, 3])

tensor2d = torch.tensor([[1, 2],
                        [3, 4]])

tensor3d = torch.tensor([[[1, 2], [3, 4]],
                          [[5, 6],  [7, 8]]])

In [3]:
# Check the inferred dtype of an integer tensor.
print(tensor1d.dtype)

torch.int64


In [4]:
# Floating-point values produce a floating-point tensor by default.
floatVec = torch.tensor([1.0, 2.0, 3.0])
print(floatVec.dtype)

torch.float32


In [5]:
# Convert an integer tensor to a floating-point dtype.
floadVec = tensor1d.to(torch.float32)
print(floatVec.dtype)

torch.float32


In [6]:
# Inspect the dimensions of a two-dimensional tensor.
tensor2d = torch.tensor([[1, 2, 3],
                        [4, 5, 6]])
print(tensor2d.shape)

torch.Size([2, 3])


In [7]:
# Reshape the same six elements into three rows and two columns.
print(tensor2d.reshape(3, 2))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


In [8]:
# View the tensor with a different shape while sharing its storage when possible.
print(tensor2d.view(3, 2))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


In [9]:
# Transpose the rows and columns of the matrix.
print(tensor2d.T)

tensor([[1, 4],
        [2, 5],
        [3, 6]])


In [10]:
# Multiply the matrix by its transpose using matmul.
print(tensor2d.matmul(tensor2d.T))

tensor([[14, 32],
        [32, 77]])


In [11]:
# The @ operator is shorthand for matrix multiplication.
print(tensor2d @ tensor2d.T)

tensor([[14, 32],
        [32, 77]])


##### Listing A.2 A logistic regression forward pass

In [12]:
# Compute a binary-classification loss from a sigmoid probability.
import torch

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2])
b = torch.tensor([0.0])
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = torch.nn.functional.binary_cross_entropy(a, y)

##### Listing A.3 Computing gradients via autograd

In [13]:
import torch

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
# Enable autograd for the parameters whose gradients will be inspected.
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = torch.nn.functional.binary_cross_entropy(a, y)

grad_L_w1 = torch.autograd.grad(loss, w1, retain_graph=True)
grad_L_b = torch.autograd.grad(loss, b, retain_graph=True)

In [14]:
# Display the gradients returned directly by torch.autograd.grad.
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [15]:
# Backpropagate the loss and read gradients accumulated on the tensors.
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


##### Listing A.4 A multilayer perceptron with two hidden layers

In [16]:
# Define a small feed-forward network with two hidden layers.
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [17]:
# Instantiate the network for 50 input features and three output classes.
model = NeuralNetwork(50, 3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [18]:
# Count all parameters currently marked as trainable.
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


In [19]:
# Inspect the first layer's learned weight matrix.
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [20]:
# Check the dimensions of the first layer's weight matrix.
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [21]:
# Set a seed so the randomly initialized network weights are reproducible.
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [22]:
torch.manual_seed(123)
# Create one random input example and pass it through the network.
X = torch.rand((1, 50))
out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


In [23]:
# Disable gradient tracking for an inference-only forward pass.
with torch.no_grad():
    out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In [24]:
# Apply softmax to convert output logits into class probabilities.
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.3113, 0.3934, 0.2952]])


##### Listing A.5 Creating a small toy dataset

In [25]:
# Create a small labeled dataset with two input features and two classes.
X_train = torch.tensor([
 [-1.2, 3.1],
 [-0.9, 2.9],
 [-0.5, 2.6],
 [2.3, -1.1],
 [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([
 [-0.8, 2.8],
 [2.6, -1.6],
])
y_test = torch.tensor([0, 1])

##### Listing A.6 Defining a custom Dataset class

In [26]:
# Wrap feature and label tensors in the Dataset interface used by DataLoader.
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [27]:
# Report the number of examples available in the training dataset.
print(len(train_ds))

5


##### Listing A.7 Instantiating data loaders

In [28]:
# Create shuffled training batches and an ordered test loader.
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [29]:
# Inspect each batch of features and labels produced by the loader.
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx + 1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


##### Listing A.8 A training loader that drops the last batch

In [30]:
# Drop an incomplete final batch so every training batch has equal size.
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])


##### Listing A.9 Neural Network training in Pytorch

In [31]:
# Train the network with stochastic gradient descent and cross-entropy loss.
import torch

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.5
)

num_epochs = 3
for epoch in range(num_epochs):

    model.train()

    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)

        loss = torch.nn.functional.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
            f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
            f" | Train Loss: {loss:.2f}")

    model.eval()

Epoch: 001/003 | Batch 000/002 | Train Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train Loss: 0.00


In [32]:
# Evaluate the trained model without tracking gradients.
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [33]:
# Display class probabilities without scientific notation.
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])


In [34]:
# Choose the class with the highest predicted probability for each example.
predictions = torch.argmax(probas, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [35]:
# Compare predicted labels with the known training labels.
y_train == predictions

tensor([True, True, True, True, True])

In [36]:
# Count how many training examples were classified correctly.
torch.sum(predictions == y_train)

tensor(5)